# Exploratory Data Analysis

This notebook previews the raw data stored in each dataset and its corresponding evaluation results.

Import packages.

In [24]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from collections import Counter
from tqdm import tqdm
from utils.data import Dataset
from utils.model import load_eval_results
from utils.path import resolve_dataset_path

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Datasets

We're interested in the following datasets:
- [DS-1000](https://huggingface.co/datasets/xlangai/DS-1000)
- [MATH](https://huggingface.co/datasets/EleutherAI/hendrycks_math)
- [MMLU](https://huggingface.co/datasets/cais/mmlu) 

**Note:** we're **not** interested in Chatbot-Arena, Chatbot-Arena_NEW, and WildChat10K beacuse they only evaluate preferences between  two models on each instance.

In [25]:
datasets = [d.value for d in Dataset]
print(datasets)

['DS-1000', 'MATH', 'MMLU']


Load each dataset.

In [26]:
dataset_to_df = {}

kwargs = {
    "desc": "Loading datasets",
    "total": len(datasets),
    "unit": "dataset",
}

for dataset in tqdm(datasets, **kwargs):
    file_path = resolve_dataset_path(dataset)
    dataset_to_df[dataset] = pd.read_json(file_path)

Loading datasets: 100%|██████████| 3/3 [00:00<00:00, 27.55dataset/s]


View some of the instances in each dataset and ensure each dataset has the correct number  of instances.

In [27]:
for dataset, df in dataset_to_df.items():
    print(f"{dataset}: {len(df)} instances")
    display(df.head())

    num_instances = Dataset(dataset).num_instances
    assert (
        len(df) == num_instances
    ), f"{dataset} has {len(df)} instances, but {num_instances} were expected"

DS-1000: 1000 instances


,prompt,reference_code,metadata,code_context
0,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n return df.iloc[List]\n\n...","{'problem_id': 0, 'library_problem_id': 0, 'li...",import pandas as pd\nimport numpy as np\nimpor...
1,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n df2 = df.iloc[List].rein...","{'problem_id': 1, 'library_problem_id': 1, 'li...",import pandas as pd\nimport numpy as np\nimpor...
2,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 2, 'library_problem_id': 2, 'li...",import pandas as pd\nimport numpy as np\nimpor...
3,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 3, 'library_problem_id': 3, 'li...",import pandas as pd\nimport numpy as np\nimpor...
4,Problem:\nI have following pandas dataframe :\...,result = df.where(df.apply(lambda x: x.map...,"{'problem_id': 4, 'library_problem_id': 4, 'li...",import pandas as pd\nimport numpy as np\nimpor...


MATH: 5000 instances


,problem,level,type,solution,subset
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,algebra
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,algebra
2,Find $x$ such that $\lceil x \rceil + x = \dfr...,Level 4,Algebra,"First, we note that $x$ must be positive, sinc...",algebra
3,Evaluate $i^5+i^{-25}+i^{45}$.,Level 5,Algebra,We have $i^5 = i^4\cdot i = 1\cdot (i) = i$. ...,algebra
4,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,algebra


MMLU: 14042 instances


,question,subject,choices,answer
0,Find the degree for the given field extension ...,abstract_algebra,"[0, 4, 2, 6]",1
1,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",abstract_algebra,"[8, 2, 24, 120]",2
2,Find all zeros in the indicated finite field o...,abstract_algebra,"[0, 1, 0,1, 0,4]",3
3,Statement 1 | A factor group of a non-Abelian ...,abstract_algebra,"[True, True, False, False, True, False, False,...",1
4,Find the product of the given polynomials in t...,abstract_algebra,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",1


### DS-1000

View the "metadata" column to see if there are any potential ways to categorize the instances.

In [28]:
metadata_col = dataset_to_df["DS-1000"]["metadata"].to_list()
metadata_col[0]

{'problem_id': 0,
 'library_problem_id': 0,
 'library': 'Pandas',
 'test_case_cnt': 1,
 'perturbation_type': 'Origin',
 'perturbation_origin_id': 0}

`library` and `perturbation_type` are two potential ways to categorize the instances. Additionally, the [DS-1000 GitHub repo](https://github.com/xlang-ai/DS-1000/?tab=readme-ov-file#usage) says that the possible values for `library` are: 

| lib        | count |
|------------|-------|
| Matplotlib | 155   |
| Numpy      | 220   |
| Pandas     | 291   |
| Pytorch    | 68    |
| Scipy      | 106   |
| Sklearn    | 115   |
| Tensorflow | 45    |

View the unique values in the `library` column and their counts.

In [29]:
key = "library"
counts = Counter(d[key] for d in metadata_col)

counts_df = pd.DataFrame(
    sorted(counts.items(), key=lambda x: -x[1]),
    columns=[key, "count"],
)

print(f"Number of unique {key} values: {len(counts_df)}")
counts_df

Number of unique library values: 7


,library,count
0,Pandas,291
1,Numpy,220
2,Matplotlib,155
3,Sklearn,115
4,Scipy,106
5,Pytorch,68
6,Tensorflow,45


View the unique values in the `perturbation_type` column and their counts.

In [30]:
key = "perturbation_type"
counts = Counter(d[key] for d in metadata_col)

counts_df = pd.DataFrame(
    sorted(counts.items(), key=lambda x: -x[1]),
    columns=[key, "count"],
)

print(f"Number of unique {key} values: {len(counts_df)}")
counts_df

Number of unique perturbation_type values: 4


,perturbation_type,count
0,Origin,452
1,Semantic,234
2,Difficult-Rewrite,162
3,Surface,152


It looks like **both** `library` and `perturbation_type` are promising ways to categorize instances in DS-1000.

## Evaluation Results

Load each dataset's evaluation results.

In [31]:
dataset_to_eval_results = {}

kwargs = {
    "desc": "Loading evaluation results",
    "total": len(datasets),
    "unit": "dataset",
}

for dataset in tqdm(datasets, **kwargs):
    dataset_to_eval_results[dataset] = load_eval_results(dataset)

Loading evaluation results: 100%|██████████| 3/3 [00:00<00:00, 193.01dataset/s]


View each dataset's evaluation results.

In [32]:
for dataset, df in dataset_to_eval_results.items():
    print(f"{dataset}: {len(df)} instances")
    display(df.head())

DS-1000: 1000 instances


,deepseek-coder-6.7b-base,gpt-3.5-turbo-0613,gpt-4o-2024-08-06
0,1,0,0
1,0,1,1
2,1,0,1
3,1,1,1
4,0,1,0


MATH: 5000 instances


,dart-math-llama3-8b-uniform,deepseek-math-7b-instruct,gpt-4o-mini-2024-07-18,Llama-3-8B-Instruct,Llama-3.1-8B-Instruct
0,1,1,1,1,1
1,1,1,1,0,1
2,0,1,1,1,0
3,1,0,1,0,1
4,1,1,1,1,1


MMLU: 14042 instances


,claude-3.5-haiku,gpt-3.5-turbo,gpt-4o-mini-2024-07-18,Llama-3.1-70B-Instruct,Llama-3.1-8B-Instruct,Llama-3.1-Tulu-3-70B,Llama-3.1-Tulu-3-8B,Qwen2.5-72B-Instruct,Qwen2.5-7B-Instruct
0,1,0,1,1,0,1,0,1,0
1,0,0,0,1,0,0,0,1,0
2,1,1,0,1,0,0,0,0,1
3,0,0,0,0,0,1,0,1,1
4,1,0,1,1,1,1,0,1,1
